In [1]:
import pandas as pd

In [2]:
training_data = pd.read_csv('data/v0.2/flextrack-2025-training-data-v0.2.csv', parse_dates=['Timestamp_Local'])
test_data = pd.read_csv('data/v0.2/flextrack-2025-public-test-data-v0.2-with-flags.csv', parse_dates=['Timestamp_Local'])
random_preds = pd.read_csv('data/v0.2/flextrack-2025-random-prediction-data-v0.2.csv')

In [3]:
time_col = "Timestamp_Local"
site_col = "Site"
index_cols = [site_col, time_col]
cls_features = ["Dry_Bulb_Temperature_C", "Global_Horizontal_Radiation_W/m2", "Building_Power_kW"]
reg_features = cls_features + ['Demand_Response_Flag']
target_col_classification = "Demand_Response_Capacity_kW"
target_cols = [target_col_classification]
# target_cols = ["Demand_Response_Flag", "Demand_Response_Capacity_kW"]

### Create Data Splits

In [4]:
X_train = training_data[index_cols + reg_features].copy()
y_train = training_data[index_cols + target_cols].copy()

### Create lags and rolling features

In [5]:
time_offsets = {"30min": 2, "1h": 4, "2h": 8, "6h": 4 * 6, "12h": 4 * 12, "1d": 24 * 4, "7d": 24 * 4 * 7, "m": 24 * 4 * 30, "y": 24 * 4 * 365}

In [6]:
def get_lags_features(df, lags: list = ["1h","1d","7d"], diffs: list = [1, 2, 4, 8, 12, 16]):
    stats = ['mean', 'std', 'min', 'max', 'median']
    numeric_cols = ["Dry_Bulb_Temperature_C", "Global_Horizontal_Radiation_W/m2", "Building_Power_kW"]
    lags_features = []
    df = df.sort_values(["Site", "Timestamp_Local"]).copy()
    grouped_df = df.groupby('Site')
    
    shifts = []
    for offset in lags:
        shift = grouped_df[[time_col] + numeric_cols].apply(
                lambda x: x.set_index('Timestamp_Local').shift(freq=offset),
                include_groups=False
            )
        shift.rename(columns=lambda x: f"{x}_shift_{offset}", inplace=True)
        shifts.append(shift)  

    windows_stats = []
    for window in lags:
        window_stats = []
        for stat in stats:
            window_stat = grouped_df[[time_col] + numeric_cols].apply(
                lambda x: x.set_index('Timestamp_Local').rolling(window, min_periods=time_offsets[window]).agg(stat),
                include_groups=False
            ).reset_index().set_index(['Site',"Timestamp_Local"])
            window_stat.rename(columns=lambda x: f"{x}_roll_{window}_{stat}", inplace=True)
            window_stats.append(window_stat)
        windows_stats.append(pd.concat(window_stats, axis=1))
        
    diffs_dfs = []
    for diff in diffs:
        diffs_df = grouped_df[[time_col] + numeric_cols].apply(
            lambda x: x.set_index('Timestamp_Local').diff(diff),
            include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        diffs_df.rename(columns=lambda x: f"{x}_diff_{diff}", inplace=True)
        diffs_dfs.append(diffs_df)

        pct_change_df = grouped_df[[time_col] + numeric_cols].apply(
            lambda x: x.set_index('Timestamp_Local').pct_change(diff),
            include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        pct_change_df.rename(columns=lambda x: f"{x}_pct_change_{diff}", inplace=True)
        diffs_dfs.append(pct_change_df)

        # lags_features.append(pd.concat([*shifts, *window_stats, *diffs_dfs], axis=1))
    lags_features = pd.concat([*shifts, *windows_stats, *diffs_dfs], axis=1)
    return lags_features

In [7]:
import numpy as np

def add_time_features(df, time_col='Timestamp_Local'):
    df['Minute'] = df[time_col].dt.minute
    df['Hour'] = df[time_col].dt.hour
    df['Day'] = df[time_col].dt.day
    df['Month'] = df[time_col].dt.month
    df['Year'] = df[time_col].dt.year
    df['Day_Of_Week'] = df[time_col].dt.dayofweek
    df['Is_Weekend'] = df['Day_Of_Week'].isin([5, 6]).astype(int)
    
    # Enhanced cyclical features
    df['Hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Day_Of_Week_sin'] = np.sin(2 * np.pi * df['Day_Of_Week'] / 7)
    df['Day_Of_Week_cos'] = np.cos(2 * np.pi * df['Day_Of_Week'] / 7)
    
    # Hour categories for non-linear patterns
    df['Hour_Category'] = pd.cut(df['Hour'], bins=[-1, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'])
    
    # Season definition
    df['Season'] = df['Month'].map({12:0, 1:0, 2:0, 3:1, 4:1, 5:1, 6:2, 7:2, 8:2, 9:3, 10:3, 11:3})
    return df

In [8]:
def add_custom_features(df):
    df['Temp_by_Radiation'] = df['Dry_Bulb_Temperature_C'] * df['Global_Horizontal_Radiation_W/m2']
    df['Radiation_per_Temp'] = (df['Global_Horizontal_Radiation_W/m2'] / df['Dry_Bulb_Temperature_C']).clip(lower=-1e3, upper=1e3)
    df['Power_per_Temp'] = df['Building_Power_kW'] / (df['Dry_Bulb_Temperature_C']).clip(lower=-1e3, upper=1e3)
    df['DR_Temp_Interaction'] = df['Demand_Response_Flag'] * df['Dry_Bulb_Temperature_C']
    df['DR_Power_Interaction'] = df['Demand_Response_Flag'] * df['Building_Power_kW']
    df["Dry_Bulb_Temperature_C_normalized"] = (df["Dry_Bulb_Temperature_C"] - df["Dry_Bulb_Temperature_C"].mean()) / df["Dry_Bulb_Temperature_C"].std()
    df['Global_Horizontal_Radiation_W/m2_normalized'] = (df["Global_Horizontal_Radiation_W/m2"] - df["Global_Horizontal_Radiation_W/m2"].mean()) / df["Global_Horizontal_Radiation_W/m2"].std()

    # Add binned features for non-linear relationships
    df['Temp_Binned'] = pd.cut(df['Dry_Bulb_Temperature_C'], bins=10, labels=False)
    df['Power_Binned'] = pd.cut(df['Building_Power_kW'], bins=10, labels=False)
    df['Radiation_Binned'] = pd.cut(df['Global_Horizontal_Radiation_W/m2'], bins=10, labels=False)
    
    return df

In [9]:
def add_site_stats_features(df, site_col, features, min_periods=24):
    # site_stats = df.groupby(site_col)[features].agg([
    #     'mean', 'std', 'median', 'min', 'max', 
    #     'skew', lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)
    # ])
    # site_stats.columns = [f"{col}_{stat}" for col, stat in site_stats.columns.values]
    # df = df.merge(site_stats, left_on=site_col, right_index=True)
    out = {}
    for col in features:
        grp = df.groupby(site_col)[col]
        out[f"{col}_site_exp_mean"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).mean()).droplevel(0)
        out[f"{col}_site_exp_std"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).std()).droplevel(0)
        out[f"{col}_site_exp_median"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).median()).droplevel(0)
        out[f"{col}_site_exp_skew"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).skew()).droplevel(0)
        out[f"{col}_site_exp_min"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).min()).droplevel(0)
        out[f"{col}_site_exp_max"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).max()).droplevel(0)
        out[f"{col}_site_exp_25q"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).quantile(0.25)).droplevel(0)
        out[f"{col}_site_exp_75q"] = grp.apply(lambda s: s.shift().expanding(min_periods=min_periods).quantile(0.75)).droplevel(0)

    return df.join(pd.DataFrame(out, index=df.index))

In [10]:
def add_since_last_event(df, site_col="Site", time_col="Timestamp_Local", label_col="Demand_Response_Flag"):
    # Safe for training (and for test when past flags exist)
    df = df.sort_values([site_col, time_col]).copy()
    def _since(grp):
        last_event_time = grp[time_col].where(grp[label_col].ne(0)).ffill()
        return (grp[time_col] - last_event_time).dt.total_seconds() / 60.0
    df["Minutes_Since_Last_DR"] = df.groupby(site_col, group_keys=False).apply(_since, include_groups=False).fillna(0)
    df["Days_Since_Last_DR"] = (df["Minutes_Since_Last_DR"] / 1440.0).fillna(0)
    return df

In [11]:
selected_lags = ["30min","1h", "2h", "6h", "12h", "1d", "7d"]
def preprocess(X: pd.DataFrame, selected_lags=selected_lags):
    X = X.copy()
    lags_features = get_lags_features(X.drop(columns=['Demand_Response_Flag']), selected_lags)
    X = add_site_stats_features(X, site_col, cls_features)
    X = add_custom_features(X)
    X = X.set_index(index_cols).join(lags_features).reset_index()
    X = add_time_features(X)
    X = add_since_last_event(X)
    return X

In [12]:
# selected_lags = ["30min","1h", "2h", "6h", "12h", "1d", "7d"]
# lags_features = get_lags_features(X_train[index_cols+cls_features], selected_lags)
# lags_features = get_lags_features(X_train, selected_lags)

In [13]:
# X_train = add_custom_features(X_train)
# X_train = add_site_stats_features(X_train, site_col, cls_features)

In [14]:
# X_train = pd.concat([X_train.set_index(index_cols), *lags], axis=1).sort_index().reset_index()
# X_train = X_train.set_index(index_cols).join(lags_features).reset_index()

In [15]:
# X_train = add_time_features(X_train)

In [16]:
# X_train = add_since_last_event(X_train)

In [17]:
X_train = preprocess(X_train)

In [18]:
X_train = X_train.set_index(index_cols)

y_train = y_train.set_index(index_cols)
y_train = y_train.loc[X_train.index]

In [19]:
extra_time_festures = ['Hour', 'Month', 'Year', 'Day_Of_Week']

In [20]:
cat_features = ["Is_Weekend", "Hour_Category", "Season", "Temp_Binned", "Power_Binned", "Radiation_Binned", "Demand_Response_Flag"]

### Train CatBoost Regressor
Implement CatBoost regression using existing X_train and y_train for Demand_Response_Capacity_kW, evaluate with RMSE/MAE/R2, show feature importances, and save the model.

In [21]:
# Install catboost if not available (uncomment if running in a fresh environment)
# %pip install catboost

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib
import os, json

TARGET = "Demand_Response_Capacity_kW"

# Align and drop rows with missing target
train_df = X_train.join(y_train[[TARGET]])
train_df = train_df.dropna(subset=[TARGET])

# Time-based split (last 20% per site)
# site_splits = []
# for site, grp in train_df.groupby(level=0):
#     n = len(grp)
#     cut = int(n * 0.8)
#     site_splits.append((grp.iloc[:cut], grp.iloc[cut:]))
# train_part = pd.concat([p[0] for p in site_splits])
# valid_part = pd.concat([p[1] for p in site_splits])

# Site-based split (siteC for validation)
train_sites = []
valid_sites = []
for site, grp in train_df.groupby(level=0):
   if site != "siteC":
       train_sites.append(grp)
   else:
       valid_sites.append(grp)
train_part = pd.concat(train_sites)
valid_part = pd.concat(valid_sites)

feature_cols = [c for c in train_part.columns if c not in [TARGET]]

X_tr = train_part[feature_cols]
y_tr = train_part[TARGET]
X_va = valid_part[feature_cols]
y_va = valid_part[TARGET]


train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
valid_pool = Pool(X_va, y_va, cat_features=cat_features)

model = CatBoostRegressor(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=1000,
    depth=6,
    learning_rate=0.003,
    l2_leaf_reg=6,
    random_strength=0.3,
    verbose=100,
    random_seed=321,
    colsample_bylevel=0.8,
    bagging_temperature=0.5,
    early_stopping_rounds=300,
    # task_type="GPU",
    # devices='0',
)


model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

va_preds = model.predict(X_va)

rmse = mean_squared_error(y_va, va_preds)
mae = mean_absolute_error(y_va, va_preds)
r2 = r2_score(y_va, va_preds)
print('RMSE:', rmse)
print('MAE:', mae)
print('R2:', r2)

# Top feature importances
importances = model.get_feature_importance(train_pool)
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False).head(40)
print(feat_imp)

os.makedirs('models', exist_ok=True)
model_path = 'models/catboost_regressor_capacity.cbm'
model.save_model(model_path)
print(f'Model saved to {model_path}')

# Persist with joblib
joblib.dump(model, 'models/catboost_regressor_capacity.joblib')
print('Saved joblib model.')


0:	learn: 0.0761118	test: 0.9541980	best: 0.9541980 (0)	total: 74.9ms	remaining: 1m 14s
100:	learn: 0.0644126	test: 0.9448596	best: 0.9448596 (100)	total: 3.04s	remaining: 27s
200:	learn: 0.0567317	test: 0.9369458	best: 0.9369458 (200)	total: 5.68s	remaining: 22.6s
300:	learn: 0.0510673	test: 0.9310608	best: 0.9310608 (300)	total: 8.2s	remaining: 19s
400:	learn: 0.0472404	test: 0.9269238	best: 0.9269238 (400)	total: 10.9s	remaining: 16.3s
500:	learn: 0.0444634	test: 0.9243103	best: 0.9243103 (500)	total: 13.4s	remaining: 13.4s
600:	learn: 0.0420964	test: 0.9246482	best: 0.9235220 (580)	total: 16.1s	remaining: 10.7s
700:	learn: 0.0401126	test: 0.9249216	best: 0.9235220 (580)	total: 18.6s	remaining: 7.94s
800:	learn: 0.0384906	test: 0.9249409	best: 0.9235220 (580)	total: 21.1s	remaining: 5.25s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.9235220383
bestIteration = 580

Shrink model to first 581 iterations.
RMSE: 53.98332738860489
MAE: 0.9235229199500369
R2: 0.0697

In [22]:
def nmae(y_true, y_pred, building_power):
    return (np.mean(np.abs(y_true - y_pred)) / (np.mean(building_power) + 1e-9)) * 100

In [23]:
valid_part['Demand_Response_Capacity_kW_pred'] = va_preds

In [24]:
valid_part.reset_index().groupby('Site').apply(
    lambda x: nmae(x['Demand_Response_Capacity_kW'], x['Demand_Response_Capacity_kW_pred'], x['Building_Power_kW'])
)

/tmp/ipykernel_69476/3898361415.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  valid_part.reset_index().groupby('Site').apply(


Site
siteC    0.611943
dtype: float64

### Test Predictions (Capacity Regressor)
Generate test predictions using the CatBoost Regressor trained for Demand_Response_Capacity_kW.

In [25]:
X_test = test_data[index_cols + reg_features].copy()

In [26]:
X_test = preprocess(X_test)

In [27]:
# lags_features = get_lags_features(X_test, selected_lags)
# lags_features = get_lags_features(X_test[index_cols+cls_features], selected_lags)

In [28]:
# X_test = add_custom_features(X_test)
# X_test = add_site_stats_features(X_test, site_col, cls_features)

In [29]:
# X_test = X_test.set_index(index_cols).join(lags_features).reset_index()

In [30]:
# X_test = add_time_features(X_test)

In [31]:
from catboost import CatBoostRegressor, Pool
import os, numpy as np

# Load trained regressor
reg_model_path = 'models/catboost_regressor_capacity.cbm'
assert os.path.exists(reg_model_path), 'Regressor model file not found.'
reg = CatBoostRegressor()
reg.load_model(reg_model_path)

# Prepare and align test features to the model
X_test_feat = X_test.set_index(index_cols)
model_features = reg.feature_names_
X_test_feat_aligned = X_test_feat.drop(columns=[c for c in X_test_feat.columns if c not in model_features], errors='ignore')
for f in model_features:
    if f not in X_test_feat_aligned.columns:
        X_test_feat_aligned[f] = float('nan')
X_test_feat_aligned = X_test_feat_aligned[model_features]
# cat_features = ["Is_Weekend", "Hour_Category", "Season"]
test_pool = Pool(X_test_feat_aligned, cat_features=cat_features)

# Predict capacity (clip to non-negative)
cap_preds = reg.predict(test_pool)
# cap_preds = np.clip(cap_preds, 0, None)

pred_out = X_test_feat_aligned.reset_index()[index_cols+['Demand_Response_Flag']].copy()
pred_out['Demand_Response_Capacity_kW'] = cap_preds

pred_out.head()

,Site,Timestamp_Local,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteD,2020-01-01 00:00:00,0,-1.648689e-07
1,siteD,2020-01-01 00:15:00,0,-1.706732e-07
2,siteD,2020-01-01 00:30:00,0,-1.946104e-07
3,siteD,2020-01-01 00:45:00,0,-2.062548e-07
4,siteD,2020-01-01 01:00:00,0,-2.160122e-07


In [32]:
pred_out[(pred_out['Site']=="siteD") & (pred_out['Demand_Response_Flag'] == 0.0)]['Demand_Response_Capacity_kW'] = 0.0

/tmp/ipykernel_69476/41048773.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pred_out[(pred_out['Site']=="siteD") & (pred_out['Demand_Response_Flag'] == 0.0)]['Demand_Response_Capacity_kW'] = 0.0


In [33]:
import os
import time
os.makedirs('data/submissions/main-phase', exist_ok=True)
submission_path = f'data/submissions/main-phase/{time.strftime("%Y%m%d-%H%M%S")}.csv'
pred_out.to_csv(submission_path, index=False)
print(f'Saved capacity predictions to {submission_path}')

Saved capacity predictions to data/submissions/main-phase/20250929-153414.csv
